In [5]:
import dspy
import os
lm = dspy.LM('openai/gpt-4o', api_key=os.getenv("OPENAI_API_KEY"))
dspy.configure(lm=lm)

In [6]:
lm("say hello in spanish")

['Hola']

In [12]:
# Load Schemas
import json

# Load schemas from JSON file
with open('../data/3-collection-schemas-with-search-property.json', 'r') as f:
    schemas = json.load(f)


In [17]:
from pydantic import BaseModel, field_validator, Field
from typing import Literal, Optional, Dict, List, Union, Any

class IntPropertyFilter(BaseModel):
    property_name: str
    operator: Literal["=", "<", ">", "<=", ">="]
    value: int | float

class TextPropertyFilter(BaseModel):
    property_name: str
    operator: Literal["=", "LIKE"]
    value: str

class BooleanPropertyFilter(BaseModel):
    property_name: str
    operator: Literal["=", "!="]
    value: bool

class IntAggregation(BaseModel):
    property_name: str
    metrics: Literal["COUNT", "TYPE", "MIN", "MAX", "MEAN", "MEDIAN", "MODE", "SUM"]

class TextAggregation(BaseModel):
    property_name: str
    metrics: Literal["COUNT", "TYPE", "TOP_OCCURRENCES"]
    top_occurrences_limit: Optional[int] = None

class BooleanAggregation(BaseModel):
    property_name: str
    metrics: Literal["COUNT", "TYPE", "TOTAL_TRUE", "TOTAL_FALSE", "PERCENTAGE_TRUE", "PERCENTAGE_FALSE"]

class WeaviateQuery(BaseModel):
    corresponding_natural_language_query: str
    target_collection: str
    search_query: Optional[str]
    integer_property_filter: Optional[IntPropertyFilter]
    text_property_filter: Optional[TextPropertyFilter]
    boolean_property_filter: Optional[BooleanPropertyFilter]
    integer_property_aggregation: Optional[IntAggregation]
    text_property_aggregation: Optional[TextAggregation]
    boolean_property_aggregation: Optional[BooleanAggregation]
    groupby_property: Optional[str]

In [18]:
# API descriptions
search_query_desc = (str, """Use `search_query` when you need to find the most relevant results...""")

text_property_filter_desc = (TextPropertyFilter, """Use `text_property_filter` when you need to retrieve objects...""")

int_property_filter_desc = (IntPropertyFilter, """Use `int_property_filter` when you need to return objects...""")

boolean_property_filter_desc = (BooleanPropertyFilter, """Use `boolean_property_filter` when you need to retrieve objects...""")

text_property_aggregation_desc = (TextAggregation, """Use `text_property_aggregation` when you need to compute aggregate values...""")

int_property_aggregation_desc = (IntAggregation, """Use `int_property_aggregation` when you need to perform aggregate calculations...""")

boolean_property_aggregation_desc = (BooleanAggregation, """Use `boolean_property_aggregation` when you need to aggregate data...""")

groupby_desc = (str, """Use `groupby` when you need to organize or segment results...""")

operators = [
    search_query_desc,
    text_property_filter_desc,
    int_property_filter_desc,
    boolean_property_filter_desc,
    text_property_aggregation_desc,
    int_property_aggregation_desc,
    boolean_property_aggregation_desc,
    groupby_desc
]

# Dynamic OutputFields

This is where this got tricky -- trying to create the dynamic output type from the query and update the Signature.

Tricky but not impossible.

In [61]:
from copy import deepcopy

def create_signature_with_override(
    base_signature_cls,
    field_name: str,
    new_type,
    new_field
):
    """
    Dynamically create a new Signature subclass that inherits from
    `base_signature_cls` but overrides `field_name` to have the annotation
    `new_type` and the `dspy.Field` object `new_field`.
    """

    # 1) Grab the parent class's fields.  Each is a pydantic/dspy field object.
    base_fields = base_signature_cls.model_fields  # dict of {field_name: FieldInfo}
    
    # 2) Grab the parent class's type annotations (if any).
    base_annotations = dict(getattr(base_signature_cls, '__annotations__', {}))
    
    # 3) We'll build a new class dict that explicitly re-declares all fields.
    new_class_dict = {}
    new_annotations = {}
    
    # 4) Loop over every field in the base signature
    for fname, field_info in base_fields.items():
        
        # If it's the field we want to override...
        if fname == field_name:
            # Use the new field object
            new_class_dict[fname] = new_field
            # And override the type annotation
            new_annotations[fname] = new_type
            
        else:
            # Otherwise, copy the original field over
            # (We can do a shallow or deep copy. Usually shallow is enough,
            #  but if you want to be super safe, use deepcopy.)
            new_class_dict[fname] = deepcopy(field_info)
            # Keep the original annotation (if it exists)
            if fname in base_annotations:
                new_annotations[fname] = base_annotations[fname]
    
    # 5) Attach the updated annotations
    new_class_dict['__annotations__'] = new_annotations
    
    # 6) Optionally, update the docstring to mention we overrode something
    original_doc = base_signature_cls.__doc__ or ""
    new_class_dict['__doc__'] = (
        original_doc
        + f"\n\n[Dynamically created] Overridden '{field_name}' => {new_type.__name__}"
    )

    # 7) Construct a new class name for clarity
    new_class_name = (
        f"{base_signature_cls.__name__}"
        f"__{field_name.capitalize()}As{new_type.__name__}"
    )
    
    # 8) Build and return the new class
    new_signature_cls = type(new_class_name, (base_signature_cls,), new_class_dict)
    
    return new_signature_cls

In [62]:
# Randomly sample schema and combination of APIs
import random

def sample_operator_and_schema(operators, schemas):
    """
    Randomly samples 1-3 operators and 1 schema from the provided lists
    
    Args:
        operators (list): List of operator descriptions
        schemas (list): List of database schemas
        
    Returns:
        tuple: (sampled_operators, sampled_schema) where sampled_operators is a list of 1-3 operators
    """
    num_operators = random.randint(1, 3)
    sampled_operators = random.sample(operators, num_operators)
    sampled_schema = random.choice(schemas)
    return sampled_operators, sampled_schema

def create_DynamicModel_from_operators(sampled_operators: list) -> BaseModel:
    """
    Creates a dynamic Pydantic model based on the provided operators.
    The model will inherit from WeaviateQuery but override specific fields
    to make them required based on the operators.
    
    Args:
        sampled_operators (list): List of tuples containing (operator_type, description)
        
    Returns:
        BaseModel: A new Pydantic model class with fields customized for the sampled operators
    """
    # Start with WeaviateQuery as the base
    current_model = WeaviateQuery
    
    # For each sampled operator, create a new model that overrides the relevant field
    for operator_type, _ in sampled_operators:
        field_name = None
        
        # Map operator types to their corresponding field names in WeaviateQuery
        if operator_type == str and 'search_query' in str(operator_type):
            field_name = 'search_query'
        elif operator_type == TextPropertyFilter:
            field_name = 'text_property_filter'
        elif operator_type == IntPropertyFilter:
            field_name = 'integer_property_filter'
        elif operator_type == BooleanPropertyFilter:
            field_name = 'boolean_property_filter'
        elif operator_type == TextAggregation:
            field_name = 'text_property_aggregation'
        elif operator_type == IntAggregation:
            field_name = 'integer_property_aggregation'
        elif operator_type == BooleanAggregation:
            field_name = 'boolean_property_aggregation'
        elif operator_type == str and 'groupby' in str(operator_type):
            field_name = 'groupby_property'
            
        if field_name:
            # Create a new field that's required (not Optional)
            new_field = Field(...)  # ... means required in Pydantic
            # Remove the Optional wrapper from the type
            new_type = operator_type
            # Create new model with the overridden field
            current_model = create_signature_with_override(
                current_model,
                field_name,
                new_type,
                new_field
            )
    
    return current_model

In [49]:
# Query Generator
class QueryGenerator(dspy.Signature):
    """Given a set of database operators and a schema, please generate a natural language command that would require using these operators."""

    db_schema: str = dspy.InputField()
    operator_descriptions: str = dspy.InputField()
    operators_with_values: str = dspy.OutputField()
    natural_language_query: str = dspy.OutputField()

In [10]:
# LLM-as-Judge

# Helpful for DSPy PR

In [43]:
import dspy

class BaseSignature(dspy.Signature):
    """Base signature with both inputs and outputs."""
    input_text: str = dspy.InputField()
    input_number: int = dspy.InputField()
    output_response: str = dspy.OutputField()


In [44]:
from copy import deepcopy

def create_signature_with_override(
    base_signature_cls,
    field_name: str,
    new_type,
    new_field
):
    """
    Dynamically create a new Signature subclass that inherits from
    `base_signature_cls` but overrides `field_name` to have the annotation
    `new_type` and the `dspy.Field` object `new_field`.
    """

    # 1) Grab the parent class's fields.  Each is a pydantic/dspy field object.
    base_fields = base_signature_cls.model_fields  # dict of {field_name: FieldInfo}
    
    # 2) Grab the parent class's type annotations (if any).
    base_annotations = dict(getattr(base_signature_cls, '__annotations__', {}))
    
    # 3) We'll build a new class dict that explicitly re-declares all fields.
    new_class_dict = {}
    new_annotations = {}
    
    # 4) Loop over every field in the base signature
    for fname, field_info in base_fields.items():
        
        # If it's the field we want to override...
        if fname == field_name:
            # Use the new field object
            new_class_dict[fname] = new_field
            # And override the type annotation
            new_annotations[fname] = new_type
            
        else:
            # Otherwise, copy the original field over
            # (We can do a shallow or deep copy. Usually shallow is enough,
            #  but if you want to be super safe, use deepcopy.)
            new_class_dict[fname] = deepcopy(field_info)
            # Keep the original annotation (if it exists)
            if fname in base_annotations:
                new_annotations[fname] = base_annotations[fname]
    
    # 5) Attach the updated annotations
    new_class_dict['__annotations__'] = new_annotations
    
    # 6) Optionally, update the docstring to mention we overrode something
    original_doc = base_signature_cls.__doc__ or ""
    new_class_dict['__doc__'] = (
        original_doc
        + f"\n\n[Dynamically created] Overridden '{field_name}' => {new_type.__name__}"
    )

    # 7) Construct a new class name for clarity
    new_class_name = (
        f"{base_signature_cls.__name__}"
        f"__{field_name.capitalize()}As{new_type.__name__}"
    )
    
    # 8) Build and return the new class
    new_signature_cls = type(new_class_name, (base_signature_cls,), new_class_dict)
    
    return new_signature_cls
